In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/DARPA_OPTC_EDA')
REPO_URL = 'https://github.com/farzaandalwai/DARPA_OPTC_EDA.git'
BRANCH = 'eda08'
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / '.git').exists() or git('remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('STOP: existing code directory is not the expected repository')
    if git('status', '--porcelain'):
        raise RuntimeError('STOP: code checkout has local changes; preserve them before syncing')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)


In [ ]:
CHECKED_OUT_BRANCH = git('branch', '--show-current')
CODE_HEAD = git('rev-parse', 'HEAD')
print('Checked-out branch:', CHECKED_OUT_BRANCH)
print('Exact git HEAD:', CODE_HEAD)
assert CHECKED_OUT_BRANCH == BRANCH


In [ ]:
import os
DATA_ROOT = Path('/content/drive/MyDrive/DARPA_OPTC_EDA')
assert os.path.ismount('/content/drive'), 'STOP: Google Drive is not mounted'
assert Path('/content/drive/MyDrive').is_dir(), 'STOP: MyDrive is unavailable'
assert DATA_ROOT.is_dir(), 'STOP: DARPA_OPTC_EDA artifact folder is unavailable'
print('Google Drive mounted; artifact root:', DATA_ROOT)


# SysClient0201: controlled legacy-vs-hybrid connectivity

This thin runner invokes repository logic only. No EDA10 changes, model training, feature fitting or high-degree pruning. Historical Drive source artifacts are read-only; code and adapted behavior links stay under /content.

The adapter uses explicit period roles and joins (period_role, source_node_id) to historical PROCESS node_id. It preserves literal behavior_key, all nine behavior relations, counts and min/max evidence times. Old target_node_id audits collapse but never defines new identity. Context identity is exactly period + type + literal key: no path/endpoint reinterpretation, no structure scope.

The historical 70 PROCESS features and original family metadata are checked against the feature tables. Historical CREATE topology and event counts are checked against current CREATE tables. Period-rebuilt split_structure_id supplies canonical structure_id for family-connectivity metrics; reference_full_structure_id stays unchanged as separate metadata. Both period graphs remain isolated.

Existing output directories are refused, not overwritten. If either exists from an earlier successful run, inspect its saved audit instead of rerunning the build.


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)


In [ ]:
import json
import tempfile

WORK = Path(tempfile.mkdtemp(prefix='sysclient0201_hybrid_', dir='/content'))
RUNNER = REPO / 'src/eda/prepare_sysclient0201_hybrid_graph.py'
assert RUNNER.is_file(), 'STOP: latest eda08 is missing the repository runner'
print('Local audit/adapter directory:', WORK)
print('Code version for this run:', CODE_HEAD)
ARGS = ['--data-root', str(DATA_ROOT), '--work-dir', str(WORK),
        '--behavior-links-dir', str(WORK / 'adapted_behavior'),
        '--legacy-inputs', '--context-key-policy', 'exact_behavior_key_v1',
        '--structure-id-column', 'split_structure_id']


In [ ]:
# Inspects actual historical PROCESS/behavior/CREATE schemas and roles.
# Validates the exact nine relations, counts, keys and family assignments.
# Proves original PROCESS features/metadata and CREATE topology/counts agree.
subprocess.run([sys.executable, str(RUNNER), 'legacy-audit', *ARGS], cwd=REPO, check=True)


In [ ]:
# Re-audits before publishing explicit per-period behavior links locally.
# No timestamp-based membership, old target identity, or inferred behavior.
subprocess.run([sys.executable, str(RUNNER), 'legacy-adapt', *ARGS], cwd=REPO, check=True)
adapter_audit = json.loads((WORK / 'adapted_behavior/behavior_link_adapter_audit_v1.json').read_text())
print('Adapter passed:', adapter_audit['passed'])
print('Rows in / out:', adapter_audit['source_row_count'], adapter_audit['output_row_count_by_period'])
print('Behavior event counts in / out:', adapter_audit['event_count_total_before'], adapter_audit['event_count_total_after'])
print('Context collapse:', adapter_audit['context_counts_by_period'])


In [ ]:
# Audits final feature/CREATE/adapter input compatibility and source/output
# hashes before either graph is constructed. Prints exact 70-feature policy.
audit_result = subprocess.run([sys.executable, str(RUNNER), 'audit', *ARGS], cwd=REPO)
audit_report = json.loads((WORK / 'compatibility_audit.json').read_text())
print('Final 70-feature policy passed:', audit_report['final_70_feature_policy_passed'])
print('Real-artifact compatibility audit passed:', audit_report['audit_passed'])
if audit_result.returncode != 0 or not audit_report['audit_passed']:
    for error in audit_report['errors']:
        print('BLOCKER:', error)
    print('Neither graph was built by this run. Intended outputs:', audit_report['output_paths'])
    raise RuntimeError('STOP: historical-to-hybrid input reconciliation failed')


In [ ]:
# Rechecks controlled-experiment proofs, metadata and hashes; feeds the two
# existing adapter outputs directly into separate builder configurations.
subprocess.run([sys.executable, str(RUNNER), 'prepare', *ARGS], cwd=REPO, check=True)


In [ ]:
# Verifies source/prepared hashes and refuses either existing output directory.
# Calls period_heterogeneous_graph_v1 once per period; no shared graph state.
subprocess.run([sys.executable, str(RUNNER), 'build', *ARGS], cwd=REPO, check=True)


In [ ]:
# PROCESS/context counts, forward/CREATE edges, WCCs, largest PROCESS share,
# shared context/families, max/p95 degree and top 10; old summary if readable.
subprocess.run([sys.executable, str(RUNNER), 'compare', *ARGS], cwd=REPO, check=True)
status = json.loads((WORK / 'execution_status.json').read_text())
print('Commit SHA:', CODE_HEAD)
print('Audit passed:', json.loads((WORK / 'compatibility_audit.json').read_text())['audit_passed'])
print('Completed graphs:', status['completed_periods'])
print('Output Drive paths:', audit_report['output_paths'])
print('Errors:', status['errors'])
